# 🌊 Ocean Freight Settles — Deep Dive Analysis

**Routes covered:** TD3C · TD20 · TD22 · TD25  
**Data source:** Ocean Solutions broker settles (2022 – 2025)  
**Metrics:** Price ($/MT or $/day) · Worldscale (WS)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = 'plotly_dark'

# ── Load master dataset ──────────────────────────────────────────────────────
df = pd.read_csv(
    './data/master/ocean_solutions_master.csv',
    parse_dates=['date', 'period']
)

print(f"Rows: {len(df):,}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Instruments: {sorted(df['instrument'].unique())}")
print(f"Period types: {sorted(df['periodType'].unique())}")
df.head()

Rows: 112,000
Date range: 2022-01-03 → 2025-08-19
Instruments: ['TD20', 'TD22', 'TD25', 'TD3C']
Period types: ['Balmo', 'Months', 'Spot']


,source,periodType,date,instrument,period,price,ws
0,OCEAN,Spot,2022-01-03,TD3C,2022-01-01,7.9,39.6
1,OCEAN,Balmo,2022-01-03,TD3C,2022-01-01,7.9,39.6
2,OCEAN,Months,2022-01-03,TD3C,2022-01-01,7.9,39.6
3,OCEAN,Months,2022-01-03,TD3C,2022-02-01,7.9,39.6
4,OCEAN,Months,2022-01-03,TD3C,2022-03-01,7.9,39.6


---
## 1 · Summary Statistics

In [2]:
# Per-instrument summary (Spot only for apples-to-apples)
spot = df[df['periodType'] == 'Spot'].copy()

summary = (
    spot.groupby('instrument')
    .agg(
        days    = ('date', 'nunique'),
        price_mean = ('price', 'mean'),
        price_min  = ('price', 'min'),
        price_max  = ('price', 'max'),
        price_std  = ('price', 'std'),
        ws_mean    = ('ws',    'mean'),
        ws_min     = ('ws',    'min'),
        ws_max     = ('ws',    'max'),
    )
    .round(2)
)
print("Spot settle summary (2022-2025):")
summary

Spot settle summary (2022-2025):


,days,price_mean,price_min,price_max,price_std,ws_mean,ws_min,ws_max
instrument,,,,,,,,
TD20,895,18.03,8.10,34.85,4.90,105.16,52.43,225.45
TD22,895,30.10,14.81,54.63,6.45,8.13,4.00,14.75
TD25,895,36.65,16.19,79.56,10.95,174.62,86.50,425.00
TD3C,895,12.94,6.38,25.92,3.55,59.00,32.00,129.91


---
## 2 · Spot Price History

In [3]:
COLORS = {
    'TD3C': '#00D4FF',
    'TD20': '#FF6B6B',
    'TD22': '#FFD93D',
    'TD25': '#6BCB77',
}

spot_daily = (
    spot.groupby(['date', 'instrument'])
    [['price', 'ws']]
    .mean()
    .reset_index()
)

fig = go.Figure()
for inst in ['TD3C', 'TD20', 'TD22', 'TD25']:
    d = spot_daily[spot_daily['instrument'] == inst]
    fig.add_trace(go.Scatter(
        x=d['date'], y=d['price'],
        name=inst, mode='lines',
        line=dict(color=COLORS[inst], width=1.8),
        hovertemplate='%{x|%b %d %Y}<br>Price: <b>%{y:.2f}</b><extra>' + inst + '</extra>'
    ))

fig.update_layout(
    title=dict(text='Spot Price History — All Routes', font=dict(size=20)),
    xaxis_title='Trade Date',
    yaxis_title='Price ($/MT or $/day)',
    hovermode='x unified',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
)
fig.show()

---
## 3 · Worldscale (WS) History

In [4]:
fig = go.Figure()
for inst in ['TD3C', 'TD20', 'TD22', 'TD25']:
    d = spot_daily[spot_daily['instrument'] == inst]
    fig.add_trace(go.Scatter(
        x=d['date'], y=d['ws'],
        name=inst, mode='lines',
        line=dict(color=COLORS[inst], width=1.8),
        hovertemplate='%{x|%b %d %Y}<br>WS: <b>%{y:.1f}</b><extra>' + inst + '</extra>'
    ))

fig.update_layout(
    title=dict(text='Worldscale (WS) History — All Routes', font=dict(size=20)),
    xaxis_title='Trade Date',
    yaxis_title='Worldscale',
    hovermode='x unified',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
)
fig.show()

---
## 4 · Individual Route Deep-Dives (Price + WS dual axis)

In [5]:
ROUTE_DESC = {
    'TD3C': 'TD3C — Middle East Gulf → China (VLCC)',
    'TD20': 'TD20 — West Africa → Continent (Suezmax)',
    'TD22': 'TD22 — US Gulf → China (VLCC)',
    'TD25': 'TD25 — US Gulf → UK Continent (Suezmax)',
}

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=[ROUTE_DESC[r] for r in ['TD3C','TD20','TD22','TD25']],
    shared_xaxes=True,
    vertical_spacing=0.07,
    specs=[[{'secondary_y': True}]] * 4
)

for i, inst in enumerate(['TD3C', 'TD20', 'TD22', 'TD25'], 1):
    d = spot_daily[spot_daily['instrument'] == inst]
    c = COLORS[inst]

    fig.add_trace(
        go.Scatter(x=d['date'], y=d['price'], name=f'{inst} Price',
                   mode='lines', line=dict(color=c, width=1.5),
                   showlegend=(i==1)),
        row=i, col=1, secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=d['date'], y=d['ws'], name=f'{inst} WS',
                   mode='lines', line=dict(color=c, width=1, dash='dot'),
                   opacity=0.65, showlegend=(i==1)),
        row=i, col=1, secondary_y=True
    )

    fig.update_yaxes(title_text='Price', row=i, col=1, secondary_y=False)
    fig.update_yaxes(title_text='WS', row=i, col=1, secondary_y=True)

fig.update_layout(
    title=dict(text='Route Deep-Dives — Price (solid) & WS (dotted)', font=dict(size=18)),
    height=1100,
    hovermode='x unified'
)
fig.show()

---
## 5 · Rolling Volatility (30-day)

In [6]:
fig = go.Figure()

for inst in ['TD3C', 'TD20', 'TD22', 'TD25']:
    d = (
        spot_daily[spot_daily['instrument'] == inst]
        .set_index('date')
        .sort_index()
    )
    ret = d['price'].pct_change()
    vol = ret.rolling(30).std() * np.sqrt(252) * 100  # annualised %

    fig.add_trace(go.Scatter(
        x=vol.index, y=vol.values,
        name=inst, mode='lines',
        line=dict(color=COLORS[inst], width=2),
        hovertemplate='%{x|%b %d %Y}<br>Vol: <b>%{y:.1f}%</b><extra>' + inst + '</extra>'
    ))

fig.update_layout(
    title=dict(text='30-Day Rolling Annualised Volatility (%)', font=dict(size=20)),
    xaxis_title='Date',
    yaxis_title='Annualised Volatility (%)',
    hovermode='x unified',
    height=450
)
fig.show()

---
## 6 · Correlation Heatmap (Spot Prices)

In [7]:
pivot_price = (
    spot_daily.pivot(index='date', columns='instrument', values='price')
    .dropna()
)

corr = pivot_price.corr()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale='RdBu',
    zmin=-1, zmax=1,
    text=corr.round(3).values,
    texttemplate='%{text}',
    textfont=dict(size=16),
    colorbar=dict(title='Pearson r')
))

fig.update_layout(
    title=dict(text='Spot Price Correlation Heatmap', font=dict(size=20)),
    height=450,
    width=550,
    xaxis_title='Instrument',
    yaxis_title='Instrument'
)
fig.show()

---
## 7 · Seasonal Patterns — Average Price by Month

In [8]:
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

seasonal = spot_daily.copy()
seasonal['month'] = seasonal['date'].dt.month

monthly_avg = (
    seasonal.groupby(['instrument', 'month'])['price']
    .mean()
    .reset_index()
)

fig = go.Figure()
for inst in ['TD3C', 'TD20', 'TD22', 'TD25']:
    d = monthly_avg[monthly_avg['instrument'] == inst].sort_values('month')
    fig.add_trace(go.Scatter(
        x=[MONTHS[m-1] for m in d['month']],
        y=d['price'],
        name=inst,
        mode='lines+markers',
        line=dict(color=COLORS[inst], width=2.5),
        marker=dict(size=9, color=COLORS[inst]),
        hovertemplate='%{x}<br>Avg Price: <b>%{y:.2f}</b><extra>' + inst + '</extra>'
    ))

fig.update_layout(
    title=dict(text='Seasonal Pattern — Average Spot Price by Calendar Month', font=dict(size=18)),
    xaxis_title='Month',
    yaxis_title='Average Price',
    height=450,
    hovermode='x unified'
)
fig.show()

---
## 8 · Price Distribution — Violin + Box

In [9]:
fig = go.Figure()

for inst in ['TD3C', 'TD20', 'TD22', 'TD25']:
    d = spot_daily[spot_daily['instrument'] == inst]['price'].dropna()
    fig.add_trace(go.Violin(
        y=d,
        name=inst,
        box_visible=True,
        meanline_visible=True,
        fillcolor=COLORS[inst],
        line_color='white',
        opacity=0.75,
        points='outliers'
    ))

fig.update_layout(
    title=dict(text='Spot Price Distribution by Route (Violin + Box)', font=dict(size=18)),
    yaxis_title='Price',
    height=500,
    violingap=0.15,
    violingroupgap=0.1
)
fig.show()

---
## 9 · Year-over-Year Comparison

In [10]:
yoy = spot_daily.copy()
yoy['year']     = yoy['date'].dt.year
yoy['day_of_year'] = yoy['date'].dt.dayofyear

YEAR_COLORS = {2022: '#FF6B6B', 2023: '#FFD93D', 2024: '#6BCB77', 2025: '#00D4FF'}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['TD3C', 'TD20', 'TD22', 'TD25'],
    vertical_spacing=0.12, horizontal_spacing=0.08
)

positions = [(1,1),(1,2),(2,1),(2,2)]

for (row, col), inst in zip(positions, ['TD3C','TD20','TD22','TD25']):
    d_inst = yoy[yoy['instrument'] == inst]
    for yr in sorted(d_inst['year'].unique()):
        d_yr = d_inst[d_inst['year'] == yr].sort_values('day_of_year')
        show_legend = (row == 1 and col == 1)
        fig.add_trace(
            go.Scatter(
                x=d_yr['day_of_year'], y=d_yr['price'],
                name=str(yr),
                mode='lines',
                line=dict(color=YEAR_COLORS.get(yr, '#aaa'), width=2),
                legendgroup=str(yr),
                showlegend=show_legend,
                hovertemplate=f'DoY: %{{x}}<br>Price: <b>%{{y:.2f}}</b><extra>{yr}</extra>'
            ),
            row=row, col=col
        )

fig.update_layout(
    title=dict(text='Year-over-Year Spot Price Comparison', font=dict(size=18)),
    height=650,
    hovermode='x unified'
)
fig.update_xaxes(title_text='Day of Year')
fig.update_yaxes(title_text='Price')
fig.show()

---
## 10 · Latest Forward Curve Snapshot

In [11]:
# Most recent trade date available
latest_date = df['date'].max()
print(f"Latest settle date: {latest_date.date()}")

curve_df = (
    df[
        (df['date'] == latest_date) &
        (df['periodType'] == 'Months')
    ]
    .dropna(subset=['price'])
    .sort_values('period')
)

fig = go.Figure()
for inst in ['TD3C', 'TD20', 'TD22', 'TD25']:
    d = curve_df[curve_df['instrument'] == inst]
    if d.empty:
        continue
    fig.add_trace(go.Scatter(
        x=d['period'],
        y=d['price'],
        name=inst,
        mode='lines+markers',
        line=dict(color=COLORS[inst], width=2.5),
        marker=dict(size=8),
        hovertemplate='%{x|%b %Y}<br>Price: <b>%{y:.2f}</b><extra>' + inst + '</extra>'
    ))

fig.update_layout(
    title=dict(
        text=f'Forward Curve Snapshot — {latest_date.strftime("%d %b %Y")}',
        font=dict(size=18)
    ),
    xaxis_title='Contract Month',
    yaxis_title='Price',
    height=470,
    hovermode='x unified'
)
fig.show()

Latest settle date: 2025-08-19


---
## 11 · WS Scatter — TD3C vs TD20 (coloured by year)

In [12]:
scatter_df = pivot_price.copy()
scatter_df['year'] = scatter_df.index.year
scatter_df = scatter_df.dropna(subset=['TD3C','TD20'])

fig = px.scatter(
    scatter_df.reset_index(),
    x='TD3C', y='TD20',
    color='year',
    color_continuous_scale='Turbo',
    hover_data={'date': True, 'year': False},
    opacity=0.65,
    title='TD3C vs TD20 Spot Price Scatter (coloured by year)'
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(
    height=500,
    xaxis_title='TD3C Price',
    yaxis_title='TD20 Price'
)
fig.show()

---
## 12 · Monthly Average Heat Map (WS by Route)

In [13]:
heat = spot_daily.copy()
heat['year']  = heat['date'].dt.year
heat['month'] = heat['date'].dt.month

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['TD3C WS', 'TD20 WS', 'TD22 WS', 'TD25 WS']
)

for (row, col), inst in zip([(1,1),(1,2),(2,1),(2,2)], ['TD3C','TD20','TD22','TD25']):
    d_inst = heat[heat['instrument'] == inst]
    pivot = (
        d_inst.groupby(['year','month'])['ws']
        .mean()
        .unstack('month')
    )
    pivot.columns = [MONTHS[m-1] for m in pivot.columns]

    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(),
            y=pivot.index.tolist(),
            colorscale='RdYlGn',
            showscale=(row==1 and col==2),
            colorbar=dict(title='WS', x=1.02),
            hovertemplate='Month: %{x}<br>Year: %{y}<br>WS: <b>%{z:.1f}</b><extra></extra>'
        ),
        row=row, col=col
    )

fig.update_layout(
    title=dict(text='Monthly Average WS Heatmap by Route & Year', font=dict(size=18)),
    height=580
)
fig.show()

---
## 13 · Drawdown Analysis

In [14]:
fig = go.Figure()

for inst in ['TD3C', 'TD20', 'TD22', 'TD25']:
    s = (
        spot_daily[spot_daily['instrument'] == inst]
        .set_index('date')['price']
        .sort_index()
    )
    rolling_max = s.cummax()
    drawdown = (s - rolling_max) / rolling_max * 100

    fig.add_trace(go.Scatter(
        x=drawdown.index, y=drawdown.values,
        name=inst, mode='lines', fill='tozeroy',
        line=dict(color=COLORS[inst], width=1.5),
        fillcolor=COLORS[inst].replace(')', ',0.15)').replace('rgb','rgba') if COLORS[inst].startswith('rgb') else COLORS[inst] + '26',
        hovertemplate='%{x|%b %d %Y}<br>Drawdown: <b>%{y:.1f}%</b><extra>' + inst + '</extra>'
    ))

fig.update_layout(
    title=dict(text='Spot Price Drawdown from Running Peak (%)', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='Drawdown (%)',
    hovermode='x unified',
    height=450
)
fig.show()

ValueError: 
    Invalid value of type 'builtins.str' received for the 'fillcolor' property of scatter
        Received value: '#00D4FF26'

    The 'fillcolor' property is a color and may be specified as:
      - A hex string (e.g. '#ff0000')
      - An rgb/rgba string (e.g. 'rgb(255,0,0)')
      - An hsl/hsla string (e.g. 'hsl(0,100%,50%)')
      - An hsv/hsva string (e.g. 'hsv(0,100%,100%)')
      - A named CSS color: see https://plotly.com/python/css-colors/ for a list

---
## 14 · Contango vs Backwardation — M1 vs M3 Spread

In [ ]:
months_df = df[df['periodType'] == 'Months'].copy()
months_df['month_num'] = (months_df['period'].dt.year - months_df['date'].dt.year) * 12 + \
                         (months_df['period'].dt.month - months_df['date'].dt.month) + 1

m1 = months_df[months_df['month_num'] == 1][['date','instrument','price']].rename(columns={'price':'M1'})
m3 = months_df[months_df['month_num'] == 3][['date','instrument','price']].rename(columns={'price':'M3'})

spread = m1.merge(m3, on=['date','instrument'])
spread['M1_minus_M3'] = spread['M1'] - spread['M3']  # positive = backwardation

fig = go.Figure()
for inst in ['TD3C', 'TD20', 'TD22', 'TD25']:
    d = spread[spread['instrument'] == inst].sort_values('date')
    if d.empty:
        continue
    fig.add_trace(go.Scatter(
        x=d['date'], y=d['M1_minus_M3'],
        name=inst, mode='lines',
        line=dict(color=COLORS[inst], width=1.8),
        hovertemplate='%{x|%b %d %Y}<br>M1-M3: <b>%{y:.2f}</b><extra>' + inst + '</extra>'
    ))

fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.4)
fig.add_annotation(x=spread['date'].min(), y=2, text='▲ Backwardation', showarrow=False,
                   font=dict(color='lightgreen', size=11), xanchor='left')
fig.add_annotation(x=spread['date'].min(), y=-2, text='▼ Contango', showarrow=False,
                   font=dict(color='salmon', size=11), xanchor='left')

fig.update_layout(
    title=dict(text='M1 – M3 Spread (Backwardation vs Contango)', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='M1 minus M3 Price',
    hovermode='x unified',
    height=450
)
fig.show()